In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import random
import os


SEED = 42


N_VOLTAGE_ANCHORS = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random seed fixed: {SEED}")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

l = 300e-6; wb = 1e-6; wa = 1e-6; he = 60e-6
E = 170e9; nu = 0.28; epsilon_0 = 8.854e-12
V_MAX = 8.0

E_prime  = E / (1 - nu**2)
I        = (he * wb**3) / 12
alpha    = 6 * (wa / wb)**2
gamma_xy = (l / wa)**2
beta_max = (epsilon_0 * he * (V_MAX**2) * (l**4)) / (2 * E_prime * I * wa**3)


class SirenLayer(nn.Module):
    def __init__(self, in_features, out_features, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0 = omega_0
        self.linear  = nn.Linear(in_features, out_features)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1 / self.linear.in_features,
                                             1 / self.linear.in_features)
            else:
                self.linear.weight.uniform_(-np.sqrt(6 / self.linear.in_features) / self.omega_0,
                                             np.sqrt(6 / self.linear.in_features) / self.omega_0)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class MechPINN(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=5, omega_0=30.0):
        super().__init__()
        layers = [SirenLayer(2, hidden_dim, is_first=True, omega_0=omega_0)]
        for _ in range(num_layers - 1):
            layers.append(SirenLayer(hidden_dim, hidden_dim, omega_0=omega_0))
        self.net   = nn.Sequential(*layers)
        self.final = nn.Linear(hidden_dim, 1)
        with torch.no_grad():
            self.final.weight.uniform_(-np.sqrt(6 / hidden_dim) / omega_0,
                                        np.sqrt(6 / hidden_dim) / omega_0)

    def forward(self, xi, v_star):
        net_in        = torch.cat([xi, v_star], dim=1)
        spatial_shape = (xi**2) * ((1.0 - xi)**2)
        features      = self.net(net_in)
        raw           = self.final(features)
        return (v_star**2) * spatial_shape * raw

    def forward_with_x_derivatives(self, xi, v_star):


        z0 = torch.cat([xi, v_star], dim=1)
        z1 = torch.cat([torch.ones_like(xi), torch.zeros_like(v_star)], dim=1)
        z2 = torch.zeros_like(z0)
        z3 = torch.zeros_like(z0)
        z4 = torch.zeros_like(z0)

        for layer in self.net:
            omega = layer.omega_0
            weight = layer.linear.weight
            a0 = omega * layer.linear(z0)
            a1 = omega * torch.nn.functional.linear(z1, weight, None)
            a2 = omega * torch.nn.functional.linear(z2, weight, None)
            a3 = omega * torch.nn.functional.linear(z3, weight, None)
            a4 = omega * torch.nn.functional.linear(z4, weight, None)
            sin_a = torch.sin(a0)
            cos_a = torch.cos(a0)
            z0 = sin_a
            z1 = cos_a * a1
            z2 = cos_a * a2 - sin_a * a1**2
            z3 = cos_a * a3 - 3.0 * sin_a * a1 * a2 - cos_a * a1**3
            z4 = (cos_a * a4
                  - 4.0 * sin_a * a1 * a3
                  - 3.0 * sin_a * a2**2
                  - 6.0 * cos_a * a1**2 * a2
                  + sin_a * a1**4)

        raw0 = self.final(z0)
        raw1 = torch.nn.functional.linear(z1, self.final.weight, None)
        raw2 = torch.nn.functional.linear(z2, self.final.weight, None)
        raw3 = torch.nn.functional.linear(z3, self.final.weight, None)
        raw4 = torch.nn.functional.linear(z4, self.final.weight, None)

        g0 = xi**2 * (1.0 - xi)**2
        g1 = 2.0*xi - 6.0*xi**2 + 4.0*xi**3
        g2 = 2.0 - 12.0*xi + 12.0*xi**2
        g3 = -12.0 + 24.0*xi
        g4 = torch.full_like(xi, 24.0)
        scale = v_star**2

        w0 = scale * g0 * raw0
        w1 = scale * (g1*raw0 + g0*raw1)
        w2 = scale * (g2*raw0 + 2.0*g1*raw1 + g0*raw2)
        w3 = scale * (g3*raw0 + 3.0*g2*raw1 + 3.0*g1*raw2 + g0*raw3)
        w4 = scale * (g4*raw0 + 4.0*g3*raw1 + 6.0*g2*raw2
                      + 4.0*g1*raw3 + g0*raw4)
        return w0, w1, w2, w3, w4


class ElecPINN(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=5, omega_0=30.0):
        super().__init__()
        layers = [SirenLayer(3, hidden_dim, is_first=True, omega_0=omega_0)]
        for _ in range(num_layers - 1):
            layers.append(SirenLayer(hidden_dim, hidden_dim, omega_0=omega_0))
        self.net   = nn.Sequential(*layers)
        self.final = nn.Linear(hidden_dim, 1)
        with torch.no_grad():
            self.final.weight.uniform_(-np.sqrt(6 / hidden_dim) / omega_0,
                                        np.sqrt(6 / hidden_dim) / omega_0)

    def forward(self, xi, eta, v_star):


        s_xi = xi**2 * (3.0 - 2.0 * xi)
        net_in = torch.cat([s_xi, eta, v_star], dim=1)
        features = self.net(net_in)
        raw      = self.final(features)
        return (1.0 - eta) + eta * (1.0 - eta) * raw


def compute_T_bar_exact(model_mech, v_star_value, n_quad=201):
    if n_quad % 2 == 0:
        n_quad += 1
    xi_q = torch.linspace(0.001, 0.999, n_quad, device=device).view(-1, 1)
    v_q  = torch.full((n_quad, 1), v_star_value, device=device)
    w, w_xi, _, _, _ = model_mech.forward_with_x_derivatives(xi_q, v_q)
    f    = w_xi.squeeze() ** 2
    h    = (0.999 - 0.001) / (n_quad - 1)
    wts  = torch.ones(n_quad, device=device)
    wts[1:-1:2] = 4
    wts[2:-1:2] = 2
    return (h / 3) * torch.sum(wts * f)


def compute_T_bar_batch(model_mech, v_star_m, n_quad=201):
    if n_quad % 2 == 0:
        n_quad += 1
    v_flat            = v_star_m.squeeze(1)
    unique_v, inv_idx = torch.unique(v_flat.detach(), return_inverse=True)
    M                 = unique_v.shape[0]
    xi_lin            = torch.linspace(0.001, 0.999, n_quad, device=device)
    xi_big            = xi_lin.repeat(M).unsqueeze(1)
    v_big             = unique_v.repeat_interleave(n_quad).unsqueeze(1)
    w_big, w_xi_big, _, _, _ = model_mech.forward_with_x_derivatives(xi_big, v_big)
    f_big             = (w_xi_big.squeeze(1) ** 2).view(M, n_quad)
    h_step            = (0.999 - 0.001) / (n_quad - 1)
    simp_w            = torch.ones(n_quad, device=device)
    simp_w[1:-1:2]    = 4
    simp_w[2:-1:2]    = 2
    simp_w            = simp_w * (h_step / 3)
    T_bar_unique      = (f_big * simp_w.unsqueeze(0)).sum(dim=1)
    return T_bar_unique[inv_idx].unsqueeze(1)


def compute_mech_loss(model_mech, model_elec, xi_m, v_star_m, alpha_scale=1.0):
    v_m, v_x, v_xx, v_xxx, v_xxxx = model_mech.forward_with_x_derivatives(
        xi_m, v_star_m)

    h_m = v_m + 1.0
    h_m_sq = h_m ** 2

    T_bar_per_sample = compute_T_bar_batch(model_mech, v_star_m, n_quad=201)
    T_bar_term = alpha * alpha_scale * T_bar_per_sample

    eta_m = torch.ones_like(xi_m).requires_grad_(True)
    V_norm_m = model_elec(xi_m, eta_m, v_star_m)
    V_eta_m = torch.autograd.grad(V_norm_m,eta_m,torch.ones_like(V_norm_m),create_graph=True)[0]
    beta_dynamic = beta_max * (v_star_m ** 2)
    residual_raw = (h_m_sq * v_xxxx- h_m_sq * T_bar_term * v_xx+ beta_dynamic * (V_eta_m ** 2))
    scaling_factor = 1.0 / (v_star_m ** 2 + 1e-3)
    return torch.mean((residual_raw * scaling_factor) ** 2), v_m

def compute_elec_loss(model_mech, model_elec, xi, eta, v_star):
    xi  = xi.requires_grad_(True)
    eta = eta.requires_grad_(True)
    with torch.no_grad():
        v_bar = model_mech(xi, v_star)
    h       = torch.clamp(v_bar + 1.0, min=0.1)
    V_norm  = model_elec(xi, eta, v_star)
    V_xi    = torch.autograd.grad(V_norm, xi,  torch.ones_like(V_norm), create_graph=True)[0]
    V_eta   = torch.autograd.grad(V_norm, eta, torch.ones_like(V_norm), create_graph=True)[0]
    V_y     = V_eta / h
    xi_g    = xi.detach().requires_grad_(True)
    v_bar_g = model_mech(xi_g, v_star)
    v_bar_xi= torch.autograd.grad(v_bar_g, xi_g, torch.ones_like(v_bar_g),
                                   create_graph=True)[0]
    V_x     = V_xi - eta * v_bar_xi * V_y
    V_yy    = torch.autograd.grad(V_y,  eta, torch.ones_like(V_y),  create_graph=True)[0] / h
    V_x_xi  = torch.autograd.grad(V_x,  xi,  torch.ones_like(V_x),  create_graph=True)[0]
    V_x_eta = torch.autograd.grad(V_x,  eta, torch.ones_like(V_x),  create_graph=True)[0]
    V_xx    = V_x_xi - eta * v_bar_xi * V_x_eta / h
    return torch.mean(((1.0 / gamma_xy) * V_xx + V_yy) ** 2)


def compute_pde_residual(model_mech, model_elec, voltage, n_pts=201):


    model_mech.eval(); model_elec.eval()
    xi     = torch.linspace(0.01, 0.99, n_pts, device=device).view(-1, 1).requires_grad_(True)
    v_star = torch.full((n_pts, 1), voltage / V_MAX, device=device)
    v_pred, v_x, v_xx, v_xxx, v_xxxx = model_mech.forward_with_x_derivatives(
        xi, v_star)
    T_bar  = compute_T_bar_exact(model_mech, voltage / V_MAX, n_quad=201)
    h      = v_pred + 1.0
    eta    = torch.ones_like(xi).requires_grad_(True)
    V_norm = model_elec(xi, eta, v_star)
    V_eta  = torch.autograd.grad(V_norm, eta, torch.ones_like(V_norm), create_graph=True)[0]
    beta_dyn = beta_max * (v_star ** 2)
    residual = (v_xxxx- alpha * T_bar * v_xx + beta_dyn * (V_eta / h) ** 2)
    res_val  = torch.mean(torch.abs(residual)).item()
    model_mech.train(); model_elec.train()
    return res_val


class TrainingController:
    def __init__(self, initial_voltage=2.0):
        self.current_voltage = initial_voltage
        self.voltage_step    = 0.5

        self.voltage_history        = []
        self.residual_history       = []
        self.deflection_history     = []
        self.loss_mech_history      = []
        self.loss_elec_history      = []
        self.voltage_advance_epochs = []

        self.loss_history              = []
        self.gap_history               = []
        self.epochs_at_current_voltage = 0
        self.max_epochs_per_voltage    = 20000

        self.local_gap_history = []
        self.local_loss_history = []

        self.local_res_samples        = []
        self.RESIDUAL_SAMPLE_INTERVAL = 1000

    def get_min_epochs(self):
        history = self.local_gap_history if self.local_gap_history else self.gap_history
        if not history:
            return 4000
        gap = history[-1]
        if gap < 0.45:   return 20000
        elif gap < 0.55: return 16000
        elif gap < 0.65: return 14000
        elif gap < 0.75: return 12000
        elif gap < 0.85: return 10000
        else:            return 8000

    def get_dynamic_threshold(self):

        history = self.local_gap_history if self.local_gap_history else self.gap_history
        if not history:
            return 0.02

        gap = history[-1]


        if gap < 0.35:
            return 0.005
        elif gap < 0.45:
            return 0.01
        elif gap < 0.60:
            return 0.02
        elif gap < 0.75:
            return 0.03
        else:
            return 0.04

    def is_converged(self):
        if self.epochs_at_current_voltage < self.get_min_epochs():
            return False

        window = 2000
        if len(self.local_loss_history) < window * 2:
            return False

        current_threshold = self.get_dynamic_threshold()

        recent_loss = np.mean(self.local_loss_history[-window:])
        prev_loss   = np.mean(self.local_loss_history[-2*window:-window])

        if recent_loss > 0.06:
            return False

        loss_change_rate = abs(prev_loss - recent_loss) / (prev_loss + 1e-10)
        cond_loss_stable = loss_change_rate < current_threshold

        cond_res_stable = False
        if len(self.local_res_samples) >= 4:
            recent_res = np.mean(self.local_res_samples[-2:])
            prev_res   = np.mean(self.local_res_samples[-4:-2])
            res_change_rate = abs(recent_res - prev_res) / (abs(prev_res) + 1e-10)
            cond_res_stable = res_change_rate < current_threshold

        return cond_loss_stable and cond_res_stable

    def check_residual_stop(self, res_end):
        log_res = np.log10(res_end + 1e-15)
        print(f"     log10(res) = {log_res:.3f}")
        if log_res > 1.2:
            reason = f"PDE 残差发散: log10(res)={log_res:.3f} > 1.2"
            return True, reason
        return False, ""

    def try_expand_voltage(self, model_mech, model_elec, alpha_scale):
        old_voltage = self.current_voltage
        self.voltage_advance_epochs.append((len(self.loss_history), old_voltage))

        deflection, gap = evaluate_model(model_mech, old_voltage)
        self.deflection_history.append(deflection)
        self.voltage_history.append(old_voltage)


        res_end = compute_pde_residual(model_mech, model_elec, old_voltage)
        self.residual_history.append(res_end)

        print(f"  V={old_voltage:.2f}V, w={deflection:.6f}um, "
              f"log10(res_end)={np.log10(res_end+1e-15):.3f}")


        should_stop, stop_reason = self.check_residual_stop(res_end)
        if should_stop:
            return old_voltage, True, stop_reason


        if gap < 0.45:   self.voltage_step = 0.05
        elif gap < 0.55: self.voltage_step = 0.08
        elif gap < 0.65: self.voltage_step = 0.10
        elif gap < 0.75: self.voltage_step = 0.15
        else:            self.voltage_step = 0.30

        self.current_voltage          += self.voltage_step
        self.epochs_at_current_voltage = 0


        self.local_gap_history  = []
        self.local_res_samples  = []
        self.local_loss_history = []

        print(f"\n>>> {old_voltage:.2f}V -> {self.current_voltage:.2f}V "
              f"(min_ep={self.get_min_epochs()})")

        return self.current_voltage, False, ""

    def update(self, loss, min_gap):
        self.loss_history.append(loss)
        self.gap_history.append(min_gap)
        self.local_gap_history.append(min_gap)
        self.local_loss_history.append(loss)
        self.epochs_at_current_voltage += 1

        if self.is_converged():
            return 'EXPAND'

        if self.epochs_at_current_voltage >= self.max_epochs_per_voltage:
            return 'EXPAND'

        return 'CONTINUE'

    def get_v_star_max(self):
        return self.current_voltage / V_MAX

    def get_lr_factor(self):
        history = self.local_gap_history if self.local_gap_history else self.gap_history
        if not history:
            return 1.0
        gap = history[-1]
        if gap < 0.45:   return 0.02
        elif gap < 0.55: return 0.05
        elif gap < 0.65: return 0.10
        elif gap < 0.75: return 0.30
        elif gap < 0.85: return 0.50
        else:            return 1.0

    def get_summary(self):
        return {
            'final_voltage': self.current_voltage,
            'total_epochs':  len(self.loss_history),
            'voltage_data':  list(zip(self.voltage_history,
                                      self.residual_history,
                                      self.deflection_history)),
        }


def _uniform_voltage_anchors(v_star_max, n_anchors=N_VOLTAGE_ANCHORS):


    edges = torch.linspace(0.0, v_star_max, n_anchors + 1, device=device)
    anchors = edges[:-1] + torch.rand(n_anchors, device=device) * (edges[1:] - edges[:-1])
    return anchors.view(-1, 1)
def get_samples(n_mech, n_elec, v_star_max):
    xi_m = torch.rand(n_mech, 1, device=device) * 0.98 + 0.01


    anchors = _uniform_voltage_anchors(v_star_max)
    repeats = (n_mech + anchors.shape[0] - 1) // anchors.shape[0]
    v_star_m = anchors.repeat(repeats, 1)[:n_mech]
    permutation = torch.randperm(n_mech, device=device)
    v_star_m = v_star_m[permutation]

    xi_e = torch.rand(n_elec, 1, device=device) * 0.98 + 0.01
    eta_e = torch.rand(n_elec, 1, device=device)
    v_star_e = torch.rand(n_elec, 1, device=device) * v_star_max
    return xi_m, v_star_m, xi_e, eta_e, v_star_e


def evaluate_model(model_mech, voltage):
    model_mech.eval()
    xi_test = torch.linspace(0, 1, 100, device=device).view(-1, 1)
    v_star  = torch.ones_like(xi_test) * (voltage / V_MAX)
    with torch.no_grad():
        w              = model_mech(xi_test, v_star)
        max_deflection = w.min().item() * wa * 1e6
        min_gap        = (w + 1.0).min().item()
    model_mech.train()
    return max_deflection, min_gap


def train():
    pinn_mech = MechPINN().to(device)
    pinn_elec = ElecPINN().to(device)

    base_lr  = 1e-4
    opt_mech = torch.optim.Adam(pinn_mech.parameters(), lr=base_lr)
    opt_elec = torch.optim.Adam(pinn_elec.parameters(), lr=base_lr)

    controller = TrainingController(initial_voltage=2.0)

    CHECKPOINT_PATH   = "siren_300_v9_baseline128_analytic_mech_derivatives.pth"
    MAX_EPOCHS        = 500000

    import os
    start_epoch = 0

    if os.path.exists(CHECKPOINT_PATH):
        print(f"\n have {CHECKPOINT_PATH}")
        ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        pinn_mech.load_state_dict(ckpt['mech_state'])
        pinn_elec.load_state_dict(ckpt['elec_state'])
        if 'controller_state' in ckpt:
            cs = ckpt['controller_state']
            controller.current_voltage           = cs['current_voltage']
            controller.voltage_history           = cs['voltage_history']
            controller.residual_history          = cs['residual_history']
            controller.deflection_history        = cs['deflection_history']
            controller.local_gap_history         = cs.get('local_gap_history',   [])
            controller.local_res_samples         = cs.get('local_res_samples',   [])
            controller.local_loss_history        = cs.get('local_loss_history',  [])
            controller.loss_history              = cs.get('loss_history',         [])
            controller.gap_history               = cs.get('gap_history',          [])
            controller.epochs_at_current_voltage = cs.get('epochs_at_current_voltage', 0)
            controller.loss_mech_history         = cs.get('loss_mech_history',    [])
            controller.loss_elec_history         = cs.get('loss_elec_history',    [])
            controller.voltage_advance_epochs    = cs.get('voltage_advance_epochs', [])
        if 'epoch' in ckpt:
            start_epoch = ckpt['epoch']
        print(f"   recover Epoch {start_epoch}, V={controller.current_voltage:.2f}V")
        print(f"    {len(controller.voltage_history)} points recorded")

    t0 = time.time()


    epoch = start_epoch

    while epoch < MAX_EPOCHS:
        v_star_max  = controller.get_v_star_max()
        xi_m, v_star_m, xi_e, eta_e, v_star_e = get_samples(3000, 5000, v_star_max)

        alpha_scale = 1.0

        lr_factor = controller.get_lr_factor()
        for pg in opt_mech.param_groups:
            pg['lr'] = base_lr * lr_factor
        for pg in opt_elec.param_groups:
            pg['lr'] = base_lr * lr_factor

        if epoch % 5 < 4:
            pinn_elec.requires_grad_(False)
            loss_mech, _ = compute_mech_loss(pinn_mech, pinn_elec,
                                              xi_m, v_star_m, alpha_scale)
            opt_mech.zero_grad()
            loss_mech.backward()
            torch.nn.utils.clip_grad_norm_(pinn_mech.parameters(), 1.0)
            opt_mech.step()
            pinn_elec.requires_grad_(True)
            current_loss = loss_mech.item()
            controller.loss_mech_history.append((epoch, current_loss))
        else:
            pinn_mech.requires_grad_(False)
            loss_elec = compute_elec_loss(pinn_mech, pinn_elec,
                                           xi_e, eta_e, v_star_e)
            opt_elec.zero_grad()
            loss_elec.backward()
            torch.nn.utils.clip_grad_norm_(pinn_elec.parameters(), 1.0)
            opt_elec.step()
            pinn_mech.requires_grad_(True)
            current_loss = loss_elec.item()
            controller.loss_elec_history.append((epoch, current_loss))


        if epoch % controller.RESIDUAL_SAMPLE_INTERVAL == 0 and epoch > start_epoch:
            res = compute_pde_residual(
                pinn_mech, pinn_elec, controller.current_voltage)
            controller.local_res_samples.append(res)

        _, min_gap = evaluate_model(pinn_mech, controller.current_voltage)
        action     = controller.update(current_loss, min_gap)

        if action == 'EXPAND':
            result_v, stop_flag, stop_reason = controller.try_expand_voltage(
                pinn_mech, pinn_elec, alpha_scale)

            torch.save({
                'epoch':      epoch,
                'mech_state': pinn_mech.state_dict(),
                'elec_state': pinn_elec.state_dict(),
                'controller_state': {
                    'current_voltage':           controller.current_voltage,
                    'voltage_history':           controller.voltage_history,
                    'residual_history':          controller.residual_history,
                    'deflection_history':        controller.deflection_history,
                    'local_gap_history':         controller.local_gap_history,
                    'local_res_samples':         controller.local_res_samples,
                    'local_loss_history':        controller.local_loss_history,
                    'loss_history':              controller.loss_history,
                    'gap_history':               controller.gap_history,
                    'epochs_at_current_voltage': controller.epochs_at_current_voltage,
                    'loss_mech_history':         controller.loss_mech_history,
                    'loss_elec_history':         controller.loss_elec_history,
                    'voltage_advance_epochs':    controller.voltage_advance_epochs,
                },
                'summary': controller.get_summary()
            }, CHECKPOINT_PATH)

            if stop_flag:
                print(f"\n stopped: {stop_reason}")
                break

            w, gap = evaluate_model(pinn_mech, result_v)
            print(f"   predict: w={w:.6f}um, gap={gap:.4f}")

            if gap < 0.25:
                print(f"\n: gap={gap:.4f} < 0.25")
                break

        if epoch % controller.RESIDUAL_SAMPLE_INTERVAL == 0:
            w, gap = evaluate_model(pinn_mech, controller.current_voltage)
            n_res   = len(controller.local_res_samples)
            n_loss  = len(controller.local_loss_history)
            res_str = f"{controller.local_res_samples[-1]:.3e}" \
                      if controller.local_res_samples else "N/A"


            if n_loss >= 2000:
                recent   = np.mean(controller.local_loss_history[-500:])
                previous = np.mean(controller.local_loss_history[-2000:-1500])
                loss_chg = abs(recent - previous) / (previous + 1e-10)
                cond_a = f"A={'✓' if loss_chg < 0.08 else '✗'}({loss_chg:.3f})"
            else:
                cond_a = f"A=?({n_loss}/2000)"


            if n_res >= 4:
                r_res = np.mean(controller.local_res_samples[-2:])
                p_res = np.mean(controller.local_res_samples[-4:-2])
                ratio = r_res / (p_res + 1e-15)
                cond_b = f"B={'✓' if ratio <= 1.15 else '✗'}({ratio:.2f})"
            else:
                cond_b = f"B=?({n_res}/4)"

            print(f"Epoch {epoch:6d} | V={controller.current_voltage:.2f}V | "
                  f"w={w:.6f}um | gap={gap:.4f} | loss={current_loss:.2e} | "
                  f"lr×{lr_factor:.3f} | α={alpha_scale:.2f} | "
                  f"res={res_str} | {cond_a} {cond_b} | "
                  f"local_ep={controller.epochs_at_current_voltage}"
                  f"[min={controller.get_min_epochs()}]")

        epoch += 1


    summary = controller.get_summary()
    torch.save({
        'epoch':      epoch,
        'mech_state': pinn_mech.state_dict(),
        'elec_state': pinn_elec.state_dict(),
        'controller_state': {
            'current_voltage':           controller.current_voltage,
            'voltage_history':           controller.voltage_history,
            'residual_history':          controller.residual_history,
            'deflection_history':        controller.deflection_history,
            'local_gap_history':         controller.local_gap_history,
            'local_res_samples':         controller.local_res_samples,
            'local_loss_history':        controller.local_loss_history,
            'loss_history':              controller.loss_history,
            'gap_history':               controller.gap_history,
            'epochs_at_current_voltage': controller.epochs_at_current_voltage,
            'loss_mech_history':         controller.loss_mech_history,
            'loss_elec_history':         controller.loss_elec_history,
            'voltage_advance_epochs':    controller.voltage_advance_epochs,
        },
        'summary': summary
    }, CHECKPOINT_PATH)

    print(f"\nfinish! cost: {(time.time()-t0)/60:.1f} mins")
    return pinn_mech, pinn_elec, summary

if __name__ == "__main__":
    train()

